In [3]:
import wandb
import numpy as np
import json
import re

# Exact, case-sensitive W&B tag used to select runs.
RUN_TAG = "Last"

def initialize_api():
    """Initialize the Weights & Biases API."""
    return wandb.Api()

def get_runs(api, username, project_name, tag=RUN_TAG):
    """Retrieve runs carrying the selected tag, including base models and variants."""
    project_path = f'{username}/{project_name}'
    return api.runs(project_path, filters={"tags": {"$in": [tag]}})

def collect_metrics(runs):
    """Group metrics by run names and ensure numerical values."""
    grouped_metrics = {}
    
    for run in runs:
        run_name = run.name if run.name else "Unnamed"
        
        # Collect metrics ensuring they are numerical
        metrics = {}
        # metrics['creation_date'] = run.created_at
        # try:
        #     metrics['update_date'] = run.updated_at
        # except AttributeError:
        #     print('Run has no updated_at attribute, skipping this run.', run_name)
        #     continue
        for key, value in run.summary.items():
            if isinstance(value, (int, float)):
                metrics[key] = value
            else:
                try:
                    metrics[key] = float(value)
                except (TypeError, ValueError):
                    # print(f"Skipping non-numeric metric '{key}' in run '{run_name}'")
                    continue
        
        # Group metrics by run name
        if run_name not in grouped_metrics:
            grouped_metrics[run_name] = []
        grouped_metrics[run_name].append(metrics)
    
    return grouped_metrics

def model_sort_key(run_name):
    """Dataset, model, then base / MMFF / atom details (or atom types)."""
    normalized = run_name.lower().replace("_", "-")
    dataset, _, model = normalized.partition("-")
    dataset_order = {"qm9": 0, "gdb13": 1, "zinc": 2, "zinc250k": 2}
    representation = 0
    suffix = re.search(r"-(mmff|atom-(?:details|types))(?:-v\d+)?$", model)
    if suffix:
        representation = 1 if suffix.group(1) == "mmff" else 2
        model = model[:suffix.start()]
    return (dataset_order.get(dataset, 3), dataset, model, representation, run_name)

def select_exactly_3_runs(grouped_metrics):
    """Keep model groups with exactly three runs after tag filtering."""
    selected = {}
    for run_name, metrics_list in sorted(grouped_metrics.items(), key=lambda item: model_sort_key(item[0])):
        if len(metrics_list) == 3:
            selected[run_name] = metrics_list
        else:
            print(f"Warning: Skipping '{run_name}': expected exactly 3 runs, found {len(metrics_list)}.")
    return selected

def calculate_statistics(grouped_metrics, save = False, output_filename='metrics_summary.json'):
    """Calculate mean and standard deviation for each group and save results to a JSON file."""
    summary = {}

    for run_name, metrics_list in grouped_metrics.items():
        # Initialize dictionary to hold summary statistics for the current run name
        run_summary = {}

        # Collect all keys (metrics names) from the first run of the group
        keys = metrics_list[0].keys()
        metrics_summary = {key: {'values': []} for key in keys}

        # Gather all metric values
        for metrics in metrics_list:
            for key in keys:
                if key in metrics:
                    metrics_summary[key]['values'].append(metrics[key])

        # Calculate mean and std for each metric in the group
        for key, data in metrics_summary.items():
            values = np.array(data['values'])
            mean = np.mean(values)
            std = np.std(values)
            
            # Store the calculated mean and std in the run summary
            run_summary[key] = {'mean': mean, 'std': std}

        # Add the run summary to the overall summary
        summary[run_name] = run_summary

    # Write the summary to a JSON file
    if save:
        with open(output_filename, 'w') as f:
            json.dump(summary, f, indent=4)

    return summary

def print_results(results, ACCEPTED_KEYS):
    # Iterate over the results
    for run_name in sorted(results, key=model_sort_key):
        metrics = results[run_name]
        print("-"*80)
        # Sort the metrics by the order in ACCEPTED_KEYS
        sorted_metrics = {k: metrics[k] for k in ACCEPTED_KEYS if k in metrics}
        
        # Print the sorted metrics
        for metric_name in sorted_metrics:
            values = sorted_metrics[metric_name]
            is_percentage = metric_name in {
                'test/molecular_validity', 'test/molecular_uniqueness', 'test/molecular_novelty'
            }
            scale, unit = (100, "%") if is_percentage else (1, "")
            print(f"Run: {run_name}   |   {metric_name}: "
                  f"({values['mean'] * scale:.5f}{unit}, {values['std'] * scale:.5f}{unit})")

def main(tag=RUN_TAG):
    username = 'drigoni'  # Replace with your wandb username
    project_name = 'd4-hparams'

    # retrieve results from wandb
    api = initialize_api()
    runs = get_runs(api, username, project_name, tag=tag)
    grouped_metrics = collect_metrics(runs)

    # Count tagged runs before excluding groups that do not have exactly three.
    print(f"Run counts for tag {tag!r}:")
    for run_name, metrics_list in sorted(grouped_metrics.items(), key=lambda item: model_sort_key(item[0])):
        print(f"  {run_name}: {len(metrics_list)}")
    grouped_metrics = select_exactly_3_runs(grouped_metrics)
    if not grouped_metrics:
        print(f"No models with exactly 3 runs found for tag {tag!r}.")
        return {}

    # get all metrics
    all_keys = set()
    for run_metrics in grouped_metrics.values():
        all_keys.update(run_metrics[0].keys())
    print('List of all metrics:', sorted(all_keys))

    # calculate results average and std
    results = calculate_statistics(grouped_metrics)

    # print only accepted keys
    ACCEPTED_KEYS = ['test/molecular_validity', 'test/molecular_uniqueness', 'test/molecular_novelty', 
                     'test/bond_distance', 'test/bond_distance_per_class/single', 'test/bond_distance_per_class/double', 'test/bond_distance_per_class/triple',
                     'test/edge_types_distribution/single', 'test/edge_types_distribution/double', 'test/edge_types_distribution/triple',]

    print_results(results, ACCEPTED_KEYS)
    return results
        
        

if __name__ == "__main__":
    main()

Run counts for tag 'Last':
  qm9-d4: 3
  qm9-d4-mmff-v2: 3
  qm9-d4-atom-details-v2: 3
  qm9-midi: 3
  qm9-midi-mmff-v2: 3
  qm9-midi-atom-details-v2: 3
  gdb13-d4: 3
  gdb13-d4-mmff: 3
  gdb13-d4-atom_details: 3
  gdb13-midi: 3
  gdb13-midi-mmff: 3
  gdb13-midi-atom_details: 3
List of all metrics: ['_runtime', '_step', '_timestamp', '_wandb.runtime', 'epoch', 'test/E_kl', 'test/X_kl', 'test/auxiliary_node_states_kl', 'test/bond_distance', 'test/bond_distance_per_class/aromatic', 'test/bond_distance_per_class/double', 'test/bond_distance_per_class/single', 'test/bond_distance_per_class/triple', 'test/charges_kl', 'test/denoise_accuracy_c', 'test/denoise_accuracy_e', 'test/denoise_accuracy_node_hybridization', 'test/denoise_accuracy_node_is_aromatic', 'test/denoise_accuracy_node_is_in_ring', 'test/denoise_accuracy_node_mmff_type', 'test/denoise_accuracy_x', 'test/denoise_ce_auxiliary_node_states', 'test/denoise_ce_c', 'test/denoise_ce_e', 'test/denoise_ce_node_hybridization', 'test/deno

In [2]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def get_atom_info(molecule):
    atom_info = []
    for atom in molecule.GetAtoms():
        info = {
            # "Index": atom.GetIdx(),
            "Element": atom.GetSymbol(),
            "In Ring": atom.IsInRing(),                     # se l'atomo fa parte di un anello
            "Aromatic": atom.GetIsAromatic(),               # se l'atomo fa parte di un anello aromatico
            # "Atomic Mass": atom.GetMass(),
            "Hybridization": atom.GetHybridization().name,  # impatta il numero di legami che ha e potrebbe fare
            "Formal Charge": atom.GetFormalCharge(),        # carica formale dell'atomo
            # "Chiral Tag": atom.GetChiralTag().name,         # indica la chiralità dell'atomo
            # "Isotope": atom.GetIsotope(),                   # uno stesso atomo può avere isotopi diversi (es. C12, C13, C14). Stessi protoni ma più neuroni
            
            # "Ring Size": atom.IsInRingSize(5) or atom.IsInRingSize(6), # se l'atomo fa parte di un anello di dimensione 5 o 6
            # "Degree": atom.GetDegree(),
            # "Valence": atom.GetTotalValence(),
            # "Implicit Hs": atom.GetNumImplicitHs(),
            # "Explicit Hs": atom.GetNumExplicitHs(),
            # "Total Num Electrons": atom.GetTotalNumHs() + atom.GetTotalValence(),
            # "Radical Electrons": atom.GetNumRadicalElectrons(),
        }
        atom_info.append(info)
    return atom_info

def get_bond_info(molecule):
    bond_info = []
    for bond in molecule.GetBonds():
        info = {
            "Bond Type": bond.GetBondType().name,
            "Bond Order": int(bond.GetBondTypeAsDouble()),
            "Is Aromatic": bond.GetIsAromatic(),
            # "Begin Atom Index": bond.GetBeginAtomIdx(),
            # "End Atom Index": bond.GetEndAtomIdx(),
            # "Is Conjugated": bond.GetIsConjugated(),
        }
        bond_info.append(info)
    return bond_info

def print_molecule_info(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    molecule = Chem.RemoveHs(molecule)  # Aggiungi idrogeni espliciti
    Chem.SanitizeMol(molecule)
    Chem.Kekulize(molecule, clearAromaticFlags=False)
    if molecule is None:
        print("Invalid SMILES string.")
        return

    print("Atom Information:")
    atom_info = get_atom_info(molecule)
    for info in atom_info:
        print(info)

    print("\nBond Information:")
    bond_info = get_bond_info(molecule)
    for info in bond_info:
        print(info)

# Example usage
smiles = "C1=CC=CC=C1"  # Benzene
smiles = 'c1ccccc1'
# smiles = "C1CCCCC1"  # Cyclohexane
print_molecule_info(smiles)

Atom Information:
{'Element': 'C', 'In Ring': True, 'Aromatic': True, 'Hybridization': 'SP2', 'Formal Charge': 0}
{'Element': 'C', 'In Ring': True, 'Aromatic': True, 'Hybridization': 'SP2', 'Formal Charge': 0}
{'Element': 'C', 'In Ring': True, 'Aromatic': True, 'Hybridization': 'SP2', 'Formal Charge': 0}
{'Element': 'C', 'In Ring': True, 'Aromatic': True, 'Hybridization': 'SP2', 'Formal Charge': 0}
{'Element': 'C', 'In Ring': True, 'Aromatic': True, 'Hybridization': 'SP2', 'Formal Charge': 0}
{'Element': 'C', 'In Ring': True, 'Aromatic': True, 'Hybridization': 'SP2', 'Formal Charge': 0}

Bond Information:
{'Bond Type': 'DOUBLE', 'Bond Order': 2, 'Is Aromatic': True}
{'Bond Type': 'SINGLE', 'Bond Order': 1, 'Is Aromatic': True}
{'Bond Type': 'DOUBLE', 'Bond Order': 2, 'Is Aromatic': True}
{'Bond Type': 'SINGLE', 'Bond Order': 1, 'Is Aromatic': True}
{'Bond Type': 'DOUBLE', 'Bond Order': 2, 'Is Aromatic': True}
{'Bond Type': 'SINGLE', 'Bond Order': 1, 'Is Aromatic': True}
